# Beta-VAE sweep winner

Best `(base, depth, lr, weight_decay, beta)` config from the joint Stage
1 + Stage 2 sweep applied across the d-sweep. **Part A** trains on all
804 ages (in-sample); **Part B** trains on the block-stride 80/10/10
split, early-stops on `val_total_loss`, reports on test. Each Part runs
a POD baseline, a VAE d-sweep, a focus-d eval at `D_FOCUS`, and a deep
dive of the d=2 latent.

Part B additionally hosts the VAE-specific generative diagnostics at
d=16 (per-dim KL barplot, aggregate-posterior summary, latent traversal,
decoded N(0, I) samples) and the AE-vs-VAE head-to-head comparison.


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Anchor to the repo root so paleoreco imports and relative file paths
# resolve regardless of where Jupyter was launched from.
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from sklearn.decomposition import PCA

from paleoreco.data import (
    PaleoFieldDataset,
    apply_zscore,
    build_prior_cube,
    compute_zscore_stats,
)
from paleoreco.splits import (
    assign_event_label,
    block_stride_split,
    summarize_split,
)
from paleoreco.models.autoencoder import ConvAE, ConvBetaVAE, count_parameters
from paleoreco.train_vae import train as train_vae
from paleoreco.train_ae import set_seed
from paleoreco.eval.ae import reconstruct_split
from paleoreco.eval.vae import (
    compute_vae_diagnostics,
    latent_traversal,
    plot_loss_curves_vae,
    reconstruct_split_vae,
)
from paleoreco.eval.shared import (
    compute_pod_time_coefficients,
    compute_split_metrics,
    partition_latent_2d,
    per_cell_rmse_celsius,
    per_mode_learning_accuracy,
    plot_decoded_samples,
    plot_latent_2d,
    plot_latent_2d_with_sigma,
    plot_latent_sweep,
    plot_latent_traversal,
    plot_per_cell_rmse,
    plot_per_cluster_pod_distributions,
    plot_per_cluster_reconstructions,
    plot_per_mode_learning_curves,
    plot_recon_distribution,
    plot_reconstructions,
    pod_fit,
    pod_predict,
)

plt.rcParams["figure.dpi"] = 110


In [ ]:
SEED = 42
LATENT_DIMS = (2, 4, 8, 16, 32, 64)
D_FOCUS = 2
MAX_EPOCHS = 250
PATIENCE = 25
BATCH_SIZE = 32
KL_WARMUP_EPOCHS = 30
LOGVAR_CLAMP = (-10.0, 10.0)

# Layer-2 viz: toggle PCA on/off and its target dim. PCA off requires
# D_FOCUS == 2 (direct-latent viz, Bousquet-style replicate).
LAYER2_USE_PCA = False
LAYER2_PCA_DIM = 2
LAYER2_TAG = f"d{D_FOCUS}" + (f"_pca{LAYER2_PCA_DIM}" if LAYER2_USE_PCA else "")

# Sweep winner from 05_vae_sweep.ipynb (Stage 1 + Stage 2).
BASE_STAR = 64
DEPTH_STAR = 2
BETA_STAR = 1e-5
LR_STAR = 3e-4
WD_STAR = 1e-4

# Generative-diagnostic knobs.
N_DECODED_SAMPLES = 8
TRAVERSAL_N_DIMS = 4
TRAVERSAL_N_STEPS = 7
TRAVERSAL_REF_AGE_BP = 39_000

OUT_DIR = Path(REPO_ROOT) / "outputs"
FIG_DIR = OUT_DIR / "figures" / "06_vae_sweep_winner"
CSV_DIR = OUT_DIR / "csvs" / "06_vae_sweep_winner"
PARTA_CSV = CSV_DIR / "partA.csv"
PARTB_CSV = CSV_DIR / "partB.csv"
AE_VS_VAE_CSV = CSV_DIR / "ae_vs_vae_test.csv"
VAE_WINNER_PT = OUT_DIR / "checkpoints" / "05_vae_sweep" / "vae_sweep_winner.pt"
AE_WINNER_PT = OUT_DIR / "checkpoints" / "03_ae_arch_sweep" / "ae_sweep_winner.pt"
for d in (OUT_DIR, FIG_DIR, CSV_DIR):
    d.mkdir(parents=True, exist_ok=True)


def _snapshot_cb_factory(snapshots: list[dict]):
    """Return an epoch-callback that appends a CPU clone of the model state.

    Used at d=D_FOCUS to capture per-epoch state_dicts for the per-POD-mode
    learning curves; the closure keeps the snapshot list as a write target.
    """
    def cb(_epoch_idx, model):
        snapshots.append(
            {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        )
    return cb


In [ ]:
data = build_prior_cube(
    prior_csv=os.path.join(REPO_ROOT, "data/Prior.csv"),
    cache_path=os.path.join(REPO_ROOT, "data/cache/prior_cube.npz"),
)
cube = data["cube"]
ages = data["ages"]
lats = data["lats"]
lons = data["lons"]
valid = data["valid"]
N_AGES = len(ages)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cube: {cube.shape}  ages: [{ages.min()}, {ages.max()}] yr BP")
print(f"device: {device}")
print(f"winner: base={BASE_STAR}, depth={DEPTH_STAR}, lr={LR_STAR:.0e}, wd={WD_STAR:.0e}, beta={BETA_STAR:.0e}")


## Part A: 100% train (in-sample)


In [ ]:
all_idx_A = np.arange(N_AGES)
stats_A = compute_zscore_stats(cube, train_age_indices=all_idx_A, valid=valid)
cube_z_A = apply_zscore(cube, stats_A)
mask_A = stats_A["safe_valid"]
print(f"Part A safe_valid cells: {int(mask_A.sum())} / {mask_A.size}")

dataset_A = PaleoFieldDataset(cube_z_A, mask_A, all_idx_A)
loader_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=True)
print(f"Part A samples: {len(dataset_A)}")


### POD baseline (all-ages fit)


In [ ]:
max_k_A = max(LATENT_DIMS)
pod_basis_A = pod_fit(cube_z_A, all_idx_A, mask_A, max_k=max_k_A, random_state=SEED)

partA_pod_rows = []
for k in LATENT_DIMS:
    pod_pred = pod_predict(cube_z_A, all_idx_A, pod_basis_A, k)
    m = compute_split_metrics(cube_z_A[all_idx_A], pod_pred, mask_A)
    partA_pod_rows.append({
        "latent_dim":   k,
        "pod_mse_z":    m["mse_z"],
        "pod_rmse_z":   m["rmse_z"],
        "pod_rrmse":    m["rrmse"],
        "pod_r_squared":m["r_squared"],
        "pod_E_d":      m["E_d"],
    })
    print(f"  POD d={k:>2d}: E_d={m['E_d']:.3f}  R^2={m['r_squared']:.3f}")


### VAE d-sweep (winner config on all-ages)


In [ ]:
partA_vae_rows = []
partA_focus = {}
partA_focus_snapshots: list[dict] = []  # populated only at d=D_FOCUS

for k in LATENT_DIMS:
    print(f"\n[Part A {k:>2}d] training...")
    set_seed(SEED)
    model = ConvBetaVAE(
        latent_dim=k, base_channels=BASE_STAR, depth=DEPTH_STAR,
        logvar_clamp=LOGVAR_CLAMP,
    )
    n_params = count_parameters(model)

    # Capture per-epoch state_dicts only at d=D_FOCUS, used downstream
    # for the per-POD-mode learning curves.
    epoch_callback = _snapshot_cb_factory(partA_focus_snapshots) if k == D_FOCUS else None

    t0 = time.perf_counter()
    out = train_vae(
        model, loader_A, val_loader=None,
        mask=mask_A, zscore_std=stats_A["std"],
        beta=BETA_STAR, kl_warmup_epochs=KL_WARMUP_EPOCHS,
        lr=LR_STAR, weight_decay=WD_STAR,
        max_epochs=MAX_EPOCHS, patience=None,
        device=device, seed=SEED,
        verbose=False, progress=True,
        epoch_callback=epoch_callback,
    )
    seconds = time.perf_counter() - t0

    model.load_state_dict(out["best_state_dict"])
    truth_z, pred_z, mu_all, lv_all = reconstruct_split_vae(
        model, dataset_A, device=device, batch_size=BATCH_SIZE,
    )
    m = compute_split_metrics(truth_z, pred_z, mask_A)
    diag = compute_vae_diagnostics(mu_all, lv_all)
    partA_vae_rows.append({
        "latent_dim":         k,
        "vae_mse_z":          m["mse_z"],
        "vae_rmse_z":         m["rmse_z"],
        "vae_rrmse":          m["rrmse"],
        "vae_r_squared":      m["r_squared"],
        "vae_E_d":            m["E_d"],
        # KL columns keep the plan's val_ prefix even in Part A. The
        # value here is on the in-sample data (no val split); the column
        # name mirrors the plan's CSV schema for symmetry with Part B.
        "val_kl_total":       diag["kl_total"],
        "val_kl_per_dim_max": float(diag["kl_per_dim"].max()),
        "val_kl_per_dim_min": float(diag["kl_per_dim"].min()),
        "val_mu_norm":        diag["mu_norm"],
        "n_params":           n_params,
        "epochs_trained":     out["epochs_trained"],
    })

    # Persist combined df incrementally so an interrupt keeps prior rows.
    partA_df = pd.DataFrame(partA_vae_rows).merge(
        pd.DataFrame(partA_pod_rows), on="latent_dim", how="left",
    )
    partA_df.to_csv(PARTA_CSV, index=False)

    print(
        f"  vae_E_d={m['E_d']:.3f}  R^2={m['r_squared']:.3f}  "
        f"KL_total={diag['kl_total']:.3f}  "
        f"epochs={out['epochs_trained']}  ({seconds:.0f}s)"
    )

    if k == D_FOCUS:
        partA_focus = {
            "history":  out["history"],
            "truth_z":  truth_z,
            "pred_z":   pred_z,
            "mu":       mu_all,
            "logvar":   lv_all,
            "model":    model,
        }
        print(f"  captured {len(partA_focus_snapshots)} per-epoch snapshots")


In [ ]:
fig = plot_latent_sweep(
    latent_dims=LATENT_DIMS,
    model_E_d=[r["vae_E_d"] for r in partA_vae_rows],
    pod_E_d=[r["pod_E_d"] for r in partA_pod_rows],
    model_label=f"VAE (winner config, beta={BETA_STAR:.0e})",
    save_path=str(FIG_DIR / "partA_d_sweep_vae_vs_pod.png"),
)
plt.show()


### Focus-d eval at d=2


In [ ]:
fig = plot_loss_curves_vae(
    partA_focus["history"],
    save_path=str(FIG_DIR / f"partA_loss_curves_d{D_FOCUS}.png"),
)
plt.show()

fig = plot_reconstructions(
    partA_focus["truth_z"], partA_focus["pred_z"],
    stats_A, ages, lats, lons,
    save_path=str(FIG_DIR / f"partA_reconstructions_d{D_FOCUS}.png"),
)
plt.show()

rmse_c_A = per_cell_rmse_celsius(
    partA_focus["truth_z"], partA_focus["pred_z"], stats_A, mask_A,
)
fig = plot_per_cell_rmse(
    rmse_c_A, lats, lons,
    save_path=str(FIG_DIR / f"partA_per_cell_rmse_d{D_FOCUS}.png"),
)
plt.show()

fig = plot_recon_distribution(
    partA_focus["truth_z"], partA_focus["pred_z"], stats_A, mask_A,
    save_path=str(FIG_DIR / f"partA_recon_distribution_d{D_FOCUS}.png"),
)
plt.show()


### Layer-2 deep dive (Bousquet)


In [ ]:
# The latent / sigma we plot below are the Part A d=D_FOCUS model's full
# encoder outputs (mu and sigma per snapshot, all 804 ages). At
# D_FOCUS=2 sigma is two-dim and the band-overlay scatter is
# meaningful; for larger D_FOCUS the same call wouldn't apply.
Z_A = partA_focus["mu"]
S_A = np.exp(0.5 * partA_focus["logvar"])
print(f"Part A d={D_FOCUS} mu/sigma: {Z_A.shape} / {S_A.shape}")

if LAYER2_USE_PCA:
    pca_A = PCA(n_components=LAYER2_PCA_DIM, random_state=SEED)
    Z2_A = pca_A.fit_transform(Z_A)
    print(f"PCA({LAYER2_PCA_DIM}) explained variance ratios: {pca_A.explained_variance_ratio_}")
else:
    if Z_A.shape[1] != 2:
        raise ValueError(
            f"LAYER2_USE_PCA=False requires D_FOCUS=2; got latent dim {Z_A.shape[1]}"
        )
    Z2_A = Z_A

labels_sign_A = partition_latent_2d(Z2_A, "z2_sign")
labels_km4_A = partition_latent_2d(Z2_A, 4, random_state=SEED)
print(f"sign cluster sizes: {[int((labels_sign_A == c).sum()) for c in np.unique(labels_sign_A)]}")
print(f"km4 cluster sizes:  {[int((labels_km4_A == c).sum()) for c in np.unique(labels_km4_A)]}")


In [ ]:
event_labels = assign_event_label(ages)

fig = plot_latent_2d_with_sigma(
    Z2_A, S_A, event_labels, color_label="D-O event index (0 = between)",
    title=f"Part A ({LAYER2_TAG}): latent + sigma bands, coloured by D-O event",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_sigma_by_event.png"),
)
plt.show()

fig = plot_latent_2d_with_sigma(
    Z2_A, S_A, ages, color_label="age (yr BP)",
    title=f"Part A ({LAYER2_TAG}): latent + sigma bands, coloured by age",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_sigma_by_age.png"),
)
plt.show()

# Cluster overlays use plain plot_latent_2d: sigma bands would clash
# with the partition boundaries the plots are trying to highlight.
fig = plot_latent_2d(
    Z2_A, labels_sign_A, color_label="cluster (sign of axis 2)",
    title=f"Part A ({LAYER2_TAG}): partition by sign of axis 2",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_partition_sign.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_A, labels_km4_A, color_label="cluster (KMeans-4)",
    title=f"Part A ({LAYER2_TAG}): KMeans(4) partition",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_partition_km4.png"),
)
plt.show()


In [ ]:
pod_a_k_A = compute_pod_time_coefficients(cube_z_A, mask_A, pod_basis_A)
TOP_K = [0, 1, 2, 3, 4]

fig = plot_per_cluster_pod_distributions(
    pod_a_k_A, labels_sign_A, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_per_cluster_pod_sign.png"),
)
plt.show()

fig = plot_per_cluster_pod_distributions(
    pod_a_k_A, labels_km4_A, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_per_cluster_pod_km4.png"),
)
plt.show()


In [ ]:
# anomaly = z * std (zero-mean by construction).
cube_anomaly_A = cube_z_A * stats_A["std"][None]

fig = plot_per_cluster_reconstructions(
    cube_anomaly_A, labels_sign_A, lats, lons, mask_A,
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_per_cluster_recon_sign.png"),
)
plt.show()

fig = plot_per_cluster_reconstructions(
    cube_anomaly_A, labels_km4_A, lats, lons, mask_A,
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_per_cluster_recon_km4.png"),
)
plt.show()


In [ ]:
# Per-POD-mode learning curves: replay each per-epoch snapshot of the
# d=D_FOCUS Part A model, project the reconstruction onto the POD basis,
# and compute per-mode accuracy e_k(epoch).
replay_model_A = ConvBetaVAE(
    latent_dim=D_FOCUS, base_channels=BASE_STAR, depth=DEPTH_STAR,
    logvar_clamp=LOGVAR_CLAMP,
).to(device)
replay_model_A.eval()

n_ep_A = len(partA_focus_snapshots)
vae_a_k_per_epoch_A = np.empty((n_ep_A, N_AGES, max_k_A), dtype=np.float64)
replay_loader_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
with torch.no_grad():
    for ep, sd in enumerate(partA_focus_snapshots):
        replay_model_A.load_state_dict(sd)
        recons = []
        for batch in replay_loader_A:
            batch = batch.to(device, non_blocking=True)
            x_hat, _, _, _ = replay_model_A(batch)
            recons.append(x_hat.cpu().numpy())
        recon = np.concatenate(recons, axis=0)
        vae_a_k_per_epoch_A[ep] = compute_pod_time_coefficients(recon, mask_A, pod_basis_A)

e_k_A = per_mode_learning_accuracy(pod_a_k_A, vae_a_k_per_epoch_A)
fig = plot_per_mode_learning_curves(
    e_k_A, ks_to_show=list(range(min(10, max_k_A))),
    save_path=str(FIG_DIR / f"partA_per_mode_learning_d{D_FOCUS}.png"),
)
plt.show()

print("Part A final-epoch e_k (top 10 POD modes):")
for k in range(min(10, max_k_A)):
    print(f"  k={k:>2d}: e_k = {e_k_A[-1, k]:.3f}")


## Part B: 80/10/10 split (test reporting)


In [ ]:
split = block_stride_split(N_AGES)
print(summarize_split(ages, split))

stats_B = compute_zscore_stats(cube, train_age_indices=split["train"], valid=valid)
cube_z_B = apply_zscore(cube, stats_B)
mask_B = stats_B["safe_valid"]
print(f"\nPart B safe_valid cells: {int(mask_B.sum())} / {mask_B.size}")

train_ds_B = PaleoFieldDataset(cube_z_B, mask_B, split["train"])
val_ds_B   = PaleoFieldDataset(cube_z_B, mask_B, split["val"])
test_ds_B  = PaleoFieldDataset(cube_z_B, mask_B, split["test"])
train_loader_B = DataLoader(train_ds_B, batch_size=BATCH_SIZE, shuffle=True)
val_loader_B   = DataLoader(val_ds_B,   batch_size=BATCH_SIZE, shuffle=False)
print(f"Part B sizes: train={len(train_ds_B)}  val={len(val_ds_B)}  test={len(test_ds_B)}")


### POD baseline (train-fit, test eval)


In [ ]:
max_k_B = max(LATENT_DIMS)
pod_basis_B = pod_fit(cube_z_B, split["train"], mask_B, max_k=max_k_B, random_state=SEED)

partB_pod_rows = []
for k in LATENT_DIMS:
    pod_pred_test = pod_predict(cube_z_B, split["test"], pod_basis_B, k)
    m = compute_split_metrics(cube_z_B[split["test"]], pod_pred_test, mask_B)
    partB_pod_rows.append({
        "latent_dim":   k,
        "pod_mse_z":    m["mse_z"],
        "pod_rmse_z":   m["rmse_z"],
        "pod_rrmse":    m["rrmse"],
        "pod_r_squared":m["r_squared"],
        "pod_E_d":      m["E_d"],
    })
    print(f"  POD d={k:>2d}: test E_d={m['E_d']:.3f}  R^2={m['r_squared']:.3f}")


### VAE d-sweep (winner config, early-stop on val, test report)

At d=16 the model is loaded directly from `vae_sweep_winner.pt` (no
retrain). Other d's are trained from scratch with the winner config and
early stopping on val_total_loss. Per-epoch snapshots are captured only
at d=D_FOCUS for the per-POD-mode learning curves.


In [ ]:
vae_winner_ckpt = torch.load(str(VAE_WINNER_PT), map_location="cpu", weights_only=False)

partB_vae_rows = []
partB_focus = {}
partB_focus_snapshots: list[dict] = []
partB_winner_d16 = {}  # holds the d=16 model + diagnostics for downstream sections

for k in LATENT_DIMS:
    set_seed(SEED)
    model = ConvBetaVAE(
        latent_dim=k, base_channels=BASE_STAR, depth=DEPTH_STAR,
        logvar_clamp=LOGVAR_CLAMP,
    )
    n_params = count_parameters(model)

    if k == 16:
        # d=16 is the sweep winner; load directly, no retrain.
        print(f"\n[Part B {k:>2}d] loading from vae_sweep_winner.pt")
        model.load_state_dict(vae_winner_ckpt["state_dict"])
        model = model.to(device)
        out = None
        seconds = 0.0
        epochs_trained = vae_winner_ckpt["best_epoch"] + 1
        best_epoch = int(vae_winner_ckpt["best_epoch"])
        history = None
    else:
        print(f"\n[Part B {k:>2}d] training...")
        epoch_callback = (
            _snapshot_cb_factory(partB_focus_snapshots) if k == D_FOCUS else None
        )
        t0 = time.perf_counter()
        out = train_vae(
            model, train_loader_B, val_loader=val_loader_B,
            mask=mask_B, zscore_std=stats_B["std"],
            beta=BETA_STAR, kl_warmup_epochs=KL_WARMUP_EPOCHS,
            lr=LR_STAR, weight_decay=WD_STAR,
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            device=device, seed=SEED,
            verbose=False, progress=True,
            epoch_callback=epoch_callback,
        )
        seconds = time.perf_counter() - t0
        model.load_state_dict(out["best_state_dict"])
        epochs_trained = out["epochs_trained"]
        best_epoch = out["best_epoch"]
        history = out["history"]

    truth_z, pred_z, mu_test, lv_test = reconstruct_split_vae(
        model, test_ds_B, device=device, batch_size=BATCH_SIZE,
    )
    m = compute_split_metrics(truth_z, pred_z, mask_B)
    diag = compute_vae_diagnostics(mu_test, lv_test)
    partB_vae_rows.append({
        "latent_dim":         k,
        "vae_mse_z":          m["mse_z"],
        "vae_rmse_z":         m["rmse_z"],
        "vae_rrmse":          m["rrmse"],
        "vae_r_squared":      m["r_squared"],
        "vae_E_d":            m["E_d"],
        "val_kl_total":       diag["kl_total"],
        "val_kl_per_dim_max": float(diag["kl_per_dim"].max()),
        "val_kl_per_dim_min": float(diag["kl_per_dim"].min()),
        "val_mu_norm":        diag["mu_norm"],
        "n_params":           n_params,
        "epochs_trained":     epochs_trained,
        "best_epoch":         best_epoch,
    })

    partB_df = pd.DataFrame(partB_vae_rows).merge(
        pd.DataFrame(partB_pod_rows), on="latent_dim", how="left",
    )
    partB_df.to_csv(PARTB_CSV, index=False)

    print(
        f"  vae_E_d={m['E_d']:.3f}  R^2={m['r_squared']:.3f}  "
        f"KL_total={diag['kl_total']:.3f}  "
        f"epochs={epochs_trained}  best={best_epoch}  ({seconds:.0f}s)"
    )

    if k == D_FOCUS:
        partB_focus = {
            "history":     history,
            "truth_z":     truth_z,
            "pred_z":      pred_z,
            "mu":          mu_test,
            "logvar":      lv_test,
            "model":       model,
            "best_epoch":  best_epoch,
        }
        print(f"  captured {len(partB_focus_snapshots)} per-epoch snapshots")
    if k == 16:
        partB_winner_d16 = {
            "model":           model,
            "mu_test":         mu_test,
            "logvar_test":     lv_test,
            "kl_per_dim_test": diag["kl_per_dim"],
            "kl_total_test":   diag["kl_total"],
            "mu_norm_test":    diag["mu_norm"],
            "post_cov_diag_mean_test": diag["post_cov_diag_mean"],
            "test_metrics":    m,
        }


In [ ]:
fig = plot_latent_sweep(
    latent_dims=LATENT_DIMS,
    model_E_d=[r["vae_E_d"] for r in partB_vae_rows],
    pod_E_d=[r["pod_E_d"] for r in partB_pod_rows],
    model_label=f"VAE (winner config, beta={BETA_STAR:.0e})",
    save_path=str(FIG_DIR / "partB_d_sweep_vae_vs_pod.png"),
)
plt.show()


### Focus-d eval at d=2 (test)


In [ ]:
fig = plot_loss_curves_vae(
    partB_focus["history"], best_epoch=partB_focus["best_epoch"],
    save_path=str(FIG_DIR / f"partB_loss_curves_d{D_FOCUS}.png"),
)
plt.show()

test_size = partB_focus["truth_z"].shape[0]
sample_indices = np.linspace(0, test_size - 1, 5).astype(int).tolist()
fig = plot_reconstructions(
    partB_focus["truth_z"], partB_focus["pred_z"],
    stats_B, ages[split["test"]], lats, lons,
    sample_indices=sample_indices,
    save_path=str(FIG_DIR / f"partB_reconstructions_d{D_FOCUS}.png"),
)
plt.show()

rmse_c_B = per_cell_rmse_celsius(
    partB_focus["truth_z"], partB_focus["pred_z"], stats_B, mask_B,
)
fig = plot_per_cell_rmse(
    rmse_c_B, lats, lons,
    save_path=str(FIG_DIR / f"partB_per_cell_rmse_d{D_FOCUS}.png"),
)
plt.show()

fig = plot_recon_distribution(
    partB_focus["truth_z"], partB_focus["pred_z"], stats_B, mask_B,
    save_path=str(FIG_DIR / f"partB_recon_distribution_d{D_FOCUS}.png"),
)
plt.show()


### Generative diagnostics at d=16 (winner)

Per-dim KL barplot and aggregate-posterior summary on the test split,
then latent traversal (4 most-active dims, +/- 2 sigma about a Holocene
reference) and 8 decoded N(0, I) samples.


In [ ]:
kl_per_dim = partB_winner_d16["kl_per_dim_test"]
d_winner = kl_per_dim.shape[0]

fig, ax = plt.subplots(figsize=(8.5, 3.8), constrained_layout=True)
bars = ax.bar(np.arange(d_winner), kl_per_dim, color="C0")
ax.set_xlabel("latent dim k")
ax.set_ylabel(r"$\mathrm{KL}_k$ (nats, test)")
ax.set_title(
    f"Part B d=16 winner: per-dim KL  "
    f"(total={partB_winner_d16['kl_total_test']:.2f}, "
    f"mu_norm={partB_winner_d16['mu_norm_test']:.3f}, "
    f"post_cov_diag_mean={partB_winner_d16['post_cov_diag_mean_test']:.3f})"
)
ax.set_xticks(np.arange(d_winner))
ax.grid(True, alpha=0.3, axis="y")
fig.savefig(FIG_DIR / "partB_d16_kl_per_dim.png", bbox_inches="tight", dpi=120)
plt.show()

print("Aggregate-posterior summary (test, d=16):")
print(f"  mu_norm            = {partB_winner_d16['mu_norm_test']:.4f}  (~0 when q(z) is centred at the prior)")
print(f"  post_cov_diag_mean = {partB_winner_d16['post_cov_diag_mean_test']:.4f}  (~1 when q(z) matches N(0, I))")
print(f"  KL_total           = {partB_winner_d16['kl_total_test']:.4f}")


In [ ]:
# Latent traversal at d=16: traverse the 4 dims with the largest per-dim
# KL (most active = most likely to be visually informative). Reference
# is a train age near the timeline mid-point; per-dim sweep range comes
# from the test-set posterior std of mu.
winner_model = partB_winner_d16["model"]
train_ages = ages[split["train"]]
train_ref_local = int(np.argmin(np.abs(train_ages - TRAVERSAL_REF_AGE_BP)))
ref_age_idx = int(split["train"][train_ref_local])
ref_x = torch.from_numpy(cube_z_B[ref_age_idx]).to(device).unsqueeze(0)
ref_mask_chan = torch.from_numpy(mask_B.astype(np.float32)).to(device).unsqueeze(0).unsqueeze(0)
ref_in = torch.cat([ref_x, ref_mask_chan], dim=1)
winner_model.eval()
with torch.no_grad():
    mu_ref_t, _ = winner_model.encode(ref_in)
mu_ref = mu_ref_t.cpu().numpy()[0]
sigma_post = partB_winner_d16["mu_test"].std(axis=0, ddof=0).astype(mu_ref.dtype)
top_dims = np.argsort(-kl_per_dim)[:TRAVERSAL_N_DIMS].tolist()
print(f"reference age: {int(ages[ref_age_idx])} yr BP (target {TRAVERSAL_REF_AGE_BP})")
print(f"top-{TRAVERSAL_N_DIMS} most-active dims: {top_dims}")
print(f"sigma_post[top_dims]: {sigma_post[top_dims]}")

decoded, z_values = latent_traversal(
    winner_model, mu_ref, sigma_post, dims=top_dims,
    n_steps=TRAVERSAL_N_STEPS, device=device,
)
fig = plot_latent_traversal(
    decoded, z_values, top_dims, stats_B, lats, lons,
    save_path=str(FIG_DIR / "partB_d16_latent_traversal.png"),
)
plt.show()


In [ ]:
# Decoded N(0, I) samples: the unconditional generative baseline.
set_seed(SEED)
z_samples = torch.randn(N_DECODED_SAMPLES, winner_model.latent_dim, device=device)
with torch.no_grad():
    decoded_samples = winner_model.decode(z_samples).cpu().numpy()
fig = plot_decoded_samples(
    decoded_samples, stats_B, lats, lons,
    save_path=str(FIG_DIR / "partB_d16_decoded_samples.png"),
)
plt.show()


### Layer-2 deep dive


In [ ]:
# Encode all 804 ages through the Part B d=D_FOCUS model. The viz is
# descriptive (not part of selection), so coordinates are computed on
# all ages even though the model was trained on the train subset.
partB_focus["model"].eval()
all_idx_B = np.arange(N_AGES)
all_ds_B = PaleoFieldDataset(cube_z_B, mask_B, all_idx_B)
with torch.no_grad():
    encode_loader_B = DataLoader(all_ds_B, batch_size=BATCH_SIZE, shuffle=False)
    mu_chunks, lv_chunks = [], []
    for batch in encode_loader_B:
        batch = batch.to(device, non_blocking=True)
        mu_b, lv_b = partB_focus["model"].encode(batch)
        mu_chunks.append(mu_b.cpu().numpy())
        lv_chunks.append(lv_b.cpu().numpy())
Z_B = np.concatenate(mu_chunks, axis=0)
S_B = np.exp(0.5 * np.concatenate(lv_chunks, axis=0))
print(f"Part B d={D_FOCUS} mu/sigma (all 804 ages): {Z_B.shape} / {S_B.shape}")

if LAYER2_USE_PCA:
    pca_B = PCA(n_components=LAYER2_PCA_DIM, random_state=SEED)
    Z2_B = pca_B.fit_transform(Z_B)
    print(f"PCA({LAYER2_PCA_DIM}) explained variance ratios: {pca_B.explained_variance_ratio_}")
else:
    if Z_B.shape[1] != 2:
        raise ValueError(
            f"LAYER2_USE_PCA=False requires D_FOCUS=2; got latent dim {Z_B.shape[1]}"
        )
    Z2_B = Z_B

labels_sign_B = partition_latent_2d(Z2_B, "z2_sign")
labels_km4_B = partition_latent_2d(Z2_B, 4, random_state=SEED)
print(f"sign cluster sizes: {[int((labels_sign_B == c).sum()) for c in np.unique(labels_sign_B)]}")
print(f"km4 cluster sizes:  {[int((labels_km4_B == c).sum()) for c in np.unique(labels_km4_B)]}")


In [ ]:
fig = plot_latent_2d_with_sigma(
    Z2_B, S_B, event_labels, color_label="D-O event index (0 = between)",
    title=f"Part B ({LAYER2_TAG}): latent + sigma bands, coloured by D-O event",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_sigma_by_event.png"),
)
plt.show()

fig = plot_latent_2d_with_sigma(
    Z2_B, S_B, ages, color_label="age (yr BP)",
    title=f"Part B ({LAYER2_TAG}): latent + sigma bands, coloured by age",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_sigma_by_age.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_B, labels_sign_B, color_label="cluster (sign of axis 2)",
    title=f"Part B ({LAYER2_TAG}): partition by sign of axis 2",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_partition_sign.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_B, labels_km4_B, color_label="cluster (KMeans-4)",
    title=f"Part B ({LAYER2_TAG}): KMeans(4) partition",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_partition_km4.png"),
)
plt.show()


In [ ]:
pod_a_k_B = compute_pod_time_coefficients(cube_z_B, mask_B, pod_basis_B)

fig = plot_per_cluster_pod_distributions(
    pod_a_k_B, labels_sign_B, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_per_cluster_pod_sign.png"),
)
plt.show()

fig = plot_per_cluster_pod_distributions(
    pod_a_k_B, labels_km4_B, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_per_cluster_pod_km4.png"),
)
plt.show()


In [ ]:
cube_anomaly_B = cube_z_B * stats_B["std"][None]

fig = plot_per_cluster_reconstructions(
    cube_anomaly_B, labels_sign_B, lats, lons, mask_B,
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_per_cluster_recon_sign.png"),
)
plt.show()

fig = plot_per_cluster_reconstructions(
    cube_anomaly_B, labels_km4_B, lats, lons, mask_B,
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_per_cluster_recon_km4.png"),
)
plt.show()


In [ ]:
# Per-POD-mode learning curves: same replay-and-project pattern as Part A,
# using pod_basis_B (train-fit) on all 804 ages of cube_z_B.
replay_model_B = ConvBetaVAE(
    latent_dim=D_FOCUS, base_channels=BASE_STAR, depth=DEPTH_STAR,
    logvar_clamp=LOGVAR_CLAMP,
).to(device)
replay_model_B.eval()

n_ep_B = len(partB_focus_snapshots)
vae_a_k_per_epoch_B = np.empty((n_ep_B, N_AGES, max_k_B), dtype=np.float64)
replay_loader_B = DataLoader(all_ds_B, batch_size=BATCH_SIZE, shuffle=False)
with torch.no_grad():
    for ep, sd in enumerate(partB_focus_snapshots):
        replay_model_B.load_state_dict(sd)
        recons = []
        for batch in replay_loader_B:
            batch = batch.to(device, non_blocking=True)
            x_hat, _, _, _ = replay_model_B(batch)
            recons.append(x_hat.cpu().numpy())
        recon = np.concatenate(recons, axis=0)
        vae_a_k_per_epoch_B[ep] = compute_pod_time_coefficients(recon, mask_B, pod_basis_B)

e_k_B = per_mode_learning_accuracy(pod_a_k_B, vae_a_k_per_epoch_B)
fig = plot_per_mode_learning_curves(
    e_k_B, ks_to_show=list(range(min(10, max_k_B))),
    save_path=str(FIG_DIR / f"partB_per_mode_learning_d{D_FOCUS}.png"),
)
plt.show()

print("Part B final-epoch e_k (top 10 POD modes):")
for k in range(min(10, max_k_B)):
    print(f"  k={k:>2d}: e_k = {e_k_B[-1, k]:.3f}")


### AE-vs-VAE head-to-head (Part B, d=16)

Loads `ae_sweep_winner.pt`, re-evaluates the AE on val and test under the
Part B stats, and tabulates the five standard recon metrics plus
`kl_total` for both models. The headline cost of Gaussianisation is
`vae.val_mse_z - ae.val_mse_z` on row pair (val, ae) vs (val, vae).


In [ ]:
ae_ckpt = torch.load(str(AE_WINNER_PT), map_location="cpu", weights_only=False)
ae_model = ConvAE(**ae_ckpt["config"]).to(device)
ae_model.load_state_dict(ae_ckpt["state_dict"])

rows = []
for split_name, ds_ in (("val", val_ds_B), ("test", test_ds_B)):
    ae_truth, ae_pred = reconstruct_split(ae_model, ds_, device=device, batch_size=BATCH_SIZE)
    ae_m = compute_split_metrics(ae_truth, ae_pred, mask_B)
    rows.append({
        "model":     "ae",
        "split":     split_name,
        "mse_z":     ae_m["mse_z"],
        "rmse_z":    ae_m["rmse_z"],
        "rrmse":     ae_m["rrmse"],
        "r_squared": ae_m["r_squared"],
        "E_d":       ae_m["E_d"],
        "kl_total":  float("nan"),
    })
    vae_truth, vae_pred, vae_mu, vae_lv = reconstruct_split_vae(
        winner_model, ds_, device=device, batch_size=BATCH_SIZE,
    )
    vae_m = compute_split_metrics(vae_truth, vae_pred, mask_B)
    vae_diag = compute_vae_diagnostics(vae_mu, vae_lv)
    rows.append({
        "model":     "vae",
        "split":     split_name,
        "mse_z":     vae_m["mse_z"],
        "rmse_z":    vae_m["rmse_z"],
        "rrmse":     vae_m["rrmse"],
        "r_squared": vae_m["r_squared"],
        "E_d":       vae_m["E_d"],
        "kl_total":  vae_diag["kl_total"],
    })

ae_vs_vae_df = pd.DataFrame(rows)
ae_vs_vae_df.to_csv(AE_VS_VAE_CSV, index=False)
print(ae_vs_vae_df.to_string(index=False))

ae_val_mse = float(ae_vs_vae_df.query("model == 'ae' and split == 'val'")["mse_z"].iloc[0])
vae_val_mse = float(ae_vs_vae_df.query("model == 'vae' and split == 'val'")["mse_z"].iloc[0])
ae_test_mse = float(ae_vs_vae_df.query("model == 'ae' and split == 'test'")["mse_z"].iloc[0])
vae_test_mse = float(ae_vs_vae_df.query("model == 'vae' and split == 'test'")["mse_z"].iloc[0])
print(f"\nGaussianisation cost (val):  vae - ae = {vae_val_mse - ae_val_mse:+.5f} mse_z")
print(f"Gaussianisation cost (test): vae - ae = {vae_test_mse - ae_test_mse:+.5f} mse_z")
